In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import col, when, lit, rand, floor, concat, length, trim, to_date


class Sliver_Extract:
    try:
        #initilizing the constructor
        def __init__ (self,df):
            self.df = df
        #data cleaning activites
        def Columns_renaming(self):
            # names for changing the columns
            renames = {
                        "cst_id":"coustmer_id",
                        "cst_key":"coustmer_key",
                        "cst_firstname":"coustmer_firstname",
                        "cst_lastname":"coustmer_lastname",
                        "cst_marital_status":"marriage_status",
                        "cst_gndr":"coustmer_gender",
                        "cst_create_date":"coustmer_join_date"
            }
            for old,new in renames.items():
                #changing the column names
                self.df = self.df.withColumnRenamed(old,new)
            return self.df
        #fill null values before delting the duplicates
        def Fill_Null(self):
            #adding the random values for the null values in column coustmer_id
            self.df = self.df.withColumn(
                                             "coustmer_id",when(col("coustmer_id").isNull() | (length(col("coustmer_id")) < 5), 
                                                                (rand() * 700000).cast("int")).otherwise(col("coustmer_id")
                                                                                                         )
            )
            #adding the adding the random values for the null values in columns and some values are less than length of 10 char
            self.df = self.df.withColumn(
                                        "coustmer_key",when(
                                                            (col("coustmer_key").isNull()) | (length(col("coustmer_key").cast("string")) != 10),
                                                            concat(lit("AW000"),floor(rand()*700000).cast("int").cast("string"))
                                                            ).otherwise(col("coustmer_key"))
            )
            #adding the default values for firstname,lastname
            self.df = (
                        self.df.withColumn(
                                            "coustmer_firstname",
                                                                when(trim(col("coustmer_firstname")).isNull(), lit("sathwik"))
                                                            .otherwise(col("coustmer_firstname"))
                                                            ).withColumn(
                                                                        "coustmer_lastname",
                                                                        when(trim(col("coustmer_lastname")).isNull(), lit("Karra"))
                                                                        .otherwise(col("coustmer_lastname"))
                                                            )
            )
            # trimming the spaces
            self.df = self.df.withColumn("coustmer_lastname",
                                                 trim(col("coustmer_lastname"))
                                                 ).withColumn(
                                                     "coustmer_lastname",trim(col("coustmer_lastname"))
                                                     )
            return self.df
        def dervied_column(self):
            #dervide column and filling nulls
            self.df = self.df.withColumn(
                                        "marriage_status",when(col("marriage_status").isNull(),"single")
                                        .when(col("marriage_status") == "M","married")
                                        .otherwise("single")
                                        ).withColumn(
                                            "coustmer_gender",when(col("coustmer_gender").isNull(),"male")
                                            .when(col("coustmer_gender") == "F","female")
                                            .otherwise("male")
                                        )
            #filling the null values for join date
            self.df = self.df.withColumn(
                                            "coustmer_join_date",
                                                                when( col("coustmer_join_date").isNull(),to_date(lit("2026-01-12"))
                                                                     ).otherwise(col("coustmer_join_date"))
                                                                )
            self.df
        def save_to_delta(self,table_name = "coustmer_information"):
            self.df.write.format("delta").option("mode","append").saveAsTable(silver.table_name)
    except Exception as e:
        print("you have error at",e)
            
